In [1]:
# 1. Import libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 2. Load dataset
df = pd.read_csv("LeagueofLegendsPicks.csv")
print(df.head())
print(df.columns)

# 3. Select text column
# Change "Picks" if your column has another name
text = df["Picks"].fillna("").astype(str).str.lower()

# 4. NLP - TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(text)

# 5. Find best K
scores = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores.append(silhouette_score(X, labels))

best_k = range(2, 11)[scores.index(max(scores))]
print("Best K:", best_k)

# 6. K-Means
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)

# 7. Show results
print(df[["Picks", "Cluster"]].head(20))

# 8. Plot silhouette scores
plt.plot(range(2, 11), scores, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Choosing K")
plt.show()

# 9. Show important words in each cluster
words = tfidf.get_feature_names_out()

for i in range(best_k):
    top = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"Cluster {i}:", [words[j] for j in top])

# 10. Save results
df.to_csv("LeagueofLegendsPicks_clustered.csv", index=False)
print("Done!")

   bResult blueTopChamp blueJungleChamp blueMiddleChamp blueADCChamp  \
0        1       Irelia          RekSai            Ahri         Jinx   
1        0         Gnar          Rengar            Ahri      Caitlyn   
2        1     Renekton          Rengar            Fizz        Sivir   
3        0       Irelia        JarvanIV         Leblanc        Sivir   
4        1         Gnar        JarvanIV       Lissandra     Tristana   

  blueSupportChamp redTopChamp redJungleChamp redMiddleChamp redADCChamp  \
0            Janna        Gnar          Elise           Fizz       Sivir   
1            Leona      Irelia       JarvanIV           Azir       Corki   
2            Annie        Sion         LeeSin           Azir       Corki   
3           Thresh        Gnar           Nunu           Lulu      KogMaw   
4            Janna        Sion         RekSai           Lulu       Corki   

  redSupportChamp            top           jungle          middle  \
0          Thresh    Irelia_Gnar     RekS

KeyError: 'Picks'